# Adapter Configuration Setup
This section defines the base model and adapter paths so the notebook can build a LoRA-enhanced Ollama model reliably.

**Key Definitions:**
- `BASE_MODEL_NAME`: Foundation model used during adapter training and required at runtime.
- `LORA_MODEL_NAME`: Local tag for the adapter-backed model you will create.
- `ADAPTER_DIR`: Directory containing `adapter_model.safetensors` and `adapter_config.json`.
- `LORA_MODELFILE`: Path to the Modelfile used to register the adapter model in Ollama.

**Purpose:**
Initialize all model naming and filesystem references before any build or inference step. Keeping these values explicit and centralized prevents mismatches between training and runtime artifacts, reduces setup errors, and makes the workflow reproducible across environments.

In [ ]:
import os
from pathlib import Path

# This must be the same as what you trained your adapter against. If you trained your adapter against a different base model, change this to match that.
BASE_MODEL_NAME = "netops-llama3.2-vision"

# This should be the base model name with the adapter name appended to it.
LORA_MODEL_NAME = "<your adapter name here>"

# Path to your adapter directory. Generally, it will be a folder name containing the adapter_model.safetensors + adapter_config.json files.
ADAPTER_DIR = Path("adapter_dir") / LORA_MODEL_NAME

LORA_MODELFILE = Path("Modelfile.lora")

print(f"Using base model: {BASE_MODEL_NAME}")
print(f"Using adapter model: {LORA_MODEL_NAME}")
print(f"Adapter directory: {ADAPTER_DIR}")
print(f"LoRA model file: {LORA_MODELFILE}")


## Build Modelfile Content
This section prepares the Modelfile text that merges the base model, adapter weights, and runtime instructions.

**Key Definitions:**
- `SYSTEM_PROMPT`: High-priority instruction block that controls model behavior and response style.
- `modelfile_text`: Full Modelfile payload sent to Ollama during model creation.
- `FROM`: Modelfile directive that selects the base model.
- `ADAPTER`: Modelfile directive that points to the LoRA adapter directory.
- `PARAMETER`: Optional generation controls such as `temperature`.

**Purpose:**
Create a deterministic model definition that can be reused for local development, demos, and automation. By constructing the Modelfile programmatically, the notebook ensures the adapter is attached correctly, the system instructions are consistent, and runtime defaults are captured in one place for repeatable model behavior.

In [ ]:
# This can be a new prompt or if you already have a prompt in your BASE_MODEL_NAME, you can use that instead
SYSTEM_PROMPT = """
You are a helpful assistant for network operations. You will be given a question and a network diagram.
Use the information in the diagram to answer the question as best as you can. If you don't know the answer, say you don't know.
Always use all the information in the diagram to answer the question.
"""

modelfile_text = f"""FROM {BASE_MODEL_NAME}
ADAPTER {ADAPTER_DIR.resolve()}

# optional: you can overwrite the system prompt in the base model with a new one. If you want to keep the same system prompt as the base model, you can omit this section.
SYSTEM {SYSTEM_PROMPT!r}

# optional: you can also add additional run parameters
PARAMETER temperature 1
"""

## Register Adapter Model in Ollama
This step executes the Ollama CLI commands needed to create and validate the adapter-backed model.

**Key Definitions:**
- `run_command()`: Helper that runs shell commands and surfaces stdout, stderr, and failures.
- `ollama create`: CLI command that builds/registers a model from a Modelfile.
- `ollama list`: CLI command that verifies the model is present in the local registry.
- `subprocess.run`: Python API used to execute external commands safely.

**Purpose:**
Automate model registration so the notebook can fail fast and provide clear diagnostics when setup is incorrect. This makes the model build process reliable for repeated runs and confirms that the newly created adapter model is available before making chat API calls.

In [ ]:
import subprocess, shlex

def run_command(command: str):
    print(f"Running command: {command}")
    process = subprocess.run(shlex.split(command), capture_output=True, text=True)
    if process.stdout.strip():
        print(f"Output: {process.stdout.strip()}")
    if process.stderr.strip():
        print(f"Error: {process.stderr.strip()}")
    if process.returncode != 0:
        raise Exception(f"Command failed with return code {process.returncode}")
    return process

print("Creating LoRA modelfile...")
run_command(f"Ollama create {LORA_MODEL_NAME} -f {LORA_MODELFILE}")
run_command(f"Ollama list")

# Compare Base vs Adapter Responses
This section sends the same network-operations prompt to both models to observe behavior differences.

**Key Definitions:**
- `HOST`: Base URL for the local Ollama API service (default `http://localhost:11434`).
- `chat()`: Helper function that posts requests to `/api/chat` and returns model output.
- `prompt`: Evaluation input used to test summarization, findings, actions, and verification guidance.
- `temperature`: Generation control that influences response variability.

**Purpose:**
Validate that the adapter contributes meaningful domain-specific improvement over the base model. Running a side-by-side comparison with identical input provides a quick quality check for relevance, completeness, and actionability before using the model in operational workflows.

In [ ]:
import json, urllib.request

HOST = "http://localhost:11434"

def chat(model: str, user:str, temperature=1) -> str:
    payload = {
        "model": model,
        "stream": False,
        "messages": [
            {"role": "user", "content": user}
        ],
        "options": {
            "temperature": temperature
        },
    }
    request = urllib.request.Request(
        HOST + "/api/chat",
        data=json.dumps(payload).encode("utf-8"),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(request, timeout=180) as response:
        response_data = json.loads(response.read().decode("utf-8"))
        return response_data["choices"][0]["message"]["content"]
prompt = """
Given the following network log snippet, provide:
1) A summary of the log.
2) Key findings
3) Actions to take based on the log.
4) Verification steps to confirm the issue is resolved after taking action.

Log Snippet:
[2024-06-01 12:00:00] bgpd: neighbor 10.10.10.2 Down
[2024-06-01 12:01:00] kernel: eth0: Link is Down
[2024-06-01 12:02:00] kernel: eth0: Link is Up
[2024-06-01 12:08:00] bgpd: neighbor 10.10.10.2 Up
"""

print("===BASE MODEL RESPONSE===")
print(chat(BASE_MODEL_NAME, prompt))
print("\n===ADAPTER MODEL RESPONSE===")
print(chat(LORA_MODEL_NAME, prompt))